In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
import numpy as np

In [ ]:
df = pd.read_csv("dataset_for_dano_fuel.csv", parse_dates=["week"])

In [ ]:
df1 = df.copy()

# Заменяем пустые строки на NaN
df1 = df1.replace("", np.nan)

# Числовые колонки: age, children_cnt — заполняем медианой
for col in ["age", "children_cnt"]:
    df1[col] = df1[col].fillna(df1[col].median())

# Категориальные колонки — заполняем Unknown
for col in ["gender_cd", "education_level_cd", "marital_status_cd", "region", "bundle_name"]:
    df1[col] = df1[col].fillna("Unknown")

# Убираем job_title, слишком много уникальных значений
if "job_title" in df1.columns:
    del df1["job_title"]

# OLS-модель
model = smf.ols(
    formula="orders_cnt ~ service_fee_amt + age + liters + entries_cnt + C(platform) + C(region)",
    data=df1
).fit()

print(model.summary())

In [ ]:
# Churn-модель
df1["week"] = pd.to_datetime(df1["week"])
week_order = {w: i for i, w in enumerate(sorted(df1["week"].unique()))}
df1["week_idx"] = df1["week"].map(week_order)

last_week_idx = df1.groupby("party_rk")["week_idx"].max()
max_week_idx = df1["week_idx"].max()

df1_users = last_week_idx.to_frame("last_week_idx")
df1_users["churn"] = (df1_users["last_week_idx"] < max_week_idx).astype(int)

# Средние показатели по пользователям
user_features = df1.groupby("party_rk").agg({
    "service_fee_amt": "mean",
    "age": "mean",
    "entries_cnt": "sum",
    "orders_cnt": "sum",
    "bundle_name": lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
})

df1_model = df1_users.join(user_features)

# Логистическая регрессия
model_churn = smf.logit(
    formula="churn ~ service_fee_amt + age + entries_cnt + orders_cnt + C(bundle_name)",
    data=df1_model
).fit()

print(model_churn.summary())

In [ ]:
df2 = df.copy()

# Удаляем полностью пустые строки
df2 = df2.replace("", np.nan)

# Удаляем строки с NA в важных признаках
df2 = df2.dropna(subset=["service_fee_amt", "orders_cnt", "age", "liters", "entries_cnt"])

# Заполняем пустые значения region "Unknown"
df2["region"] = df2["region"].fillna("Unknown")

# Убираем job_title, слишком много уникальных значений
del df2["job_title"]

model = smf.ols(
    formula="orders_cnt ~ service_fee_amt + age + liters + entries_cnt + C(platform) + C(region)",
    data=df2
).fit()

print(model.summary())


In [ ]:
df2["week"] = pd.to_datetime(df2["week"])

# Создаём номер недели в отсчёте от минимума
week_order = {w: i for i, w in enumerate(sorted(df2["week"].unique()))}
df2["week_idx"] = df2["week"].map(week_order)

last_week_idx = df2.groupby("party_rk")["week_idx"].max()
max_week_idx = df2["week_idx"].max()

churn = (last_week_idx < max_week_idx)

df2_users = last_week_idx.to_frame("last_week_idx")
df2_users["churn"] = (df2_users["last_week_idx"] < max_week_idx).astype(int)

# Подмердживаем демографию и средние fee
user_features = df2.groupby("party_rk").agg({
    "service_fee_amt": "mean",
    "age": "mean",
    "entries_cnt": "sum",
    "orders_cnt": "sum",
    "bundle_name": lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
})

df2_model = df2_users.join(user_features)

import statsmodels.formula.api as smf

model_churn = smf.logit(
    formula="churn ~ service_fee_amt + age + entries_cnt + orders_cnt + C(bundle_name)",
    data=df2_model
).fit()

print(model_churn.summary())